In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
!mkdir -p ~/.kaggle
!cp drive/MyDrive/kaggle.json ~/.kaggle
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download adrianvinueza/gym-customers-features-and-churn
!unzip -q gym-customers-features-and-churn.zip -d churn_data

Dataset URL: https://www.kaggle.com/datasets/adrianvinueza/gym-customers-features-and-churn
License(s): CC-BY-NC-SA-4.0
100% 133k/133k [00:00<00:00, 54.8MB/s]



In [5]:
os.listdir('churn_data')

['gym_churn_us.csv']

In [6]:
df = pd.read_csv('churn_data/gym_churn_us.csv', encoding='latin1')
df.head()

,gender,Near_Location,Partner,Promo_friends,Phone,Contract_period,Group_visits,Age,Avg_additional_charges_total,Month_to_end_contract,Lifetime,Avg_class_frequency_total,Avg_class_frequency_current_month,Churn
0,1,1,1,1,0,6,1,29,14.227470,5.0,3,0.020398,0.000000,0
1,0,1,0,0,1,12,1,31,113.202938,12.0,7,1.922936,1.910244,0
2,0,1,1,0,1,1,0,28,129.448479,1.0,2,1.859098,1.736502,0
3,0,1,1,1,1,12,1,33,62.669863,12.0,2,3.205633,3.357215,0
4,1,1,1,1,1,1,0,26,198.362265,1.0,3,1.113884,1.120078,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   gender                             4000 non-null   int64  
 1   Near_Location                      4000 non-null   int64  
 2   Partner                            4000 non-null   int64  
 3   Promo_friends                      4000 non-null   int64  
 4   Phone                              4000 non-null   int64  
 5   Contract_period                    4000 non-null   int64  
 6   Group_visits                       4000 non-null   int64  
 7   Age                                4000 non-null   int64  
 8   Avg_additional_charges_total       4000 non-null   float64
 9   Month_to_end_contract              4000 non-null   float64
 10  Lifetime                           4000 non-null   int64  
 11  Avg_class_frequency_total          4000 non-null   float

In [8]:
df['Churn'].value_counts(normalize=True)

,proportion
Churn,
0,0.73475
1,0.26525


In [9]:
print("Dublikatlar:", df.duplicated().sum())
print("Bo'sh qiyamatlar:", df.isna().sum().sum())

binary_cols = ['gender', 'Near_Location', 'Partner', 'Promo_friends', 'Phone', 'Group_visits', 'Churn']
for c in binary_cols:
  print(c, sorted(df[c].unique()))

num_cols = ["Age", 'Lifetime', 'Contract_period', 'Month_to_end_contract', "Avg_additional_charges_total", 'Avg_class_frequency_total', 'Avg_class_frequency_current_month']
print(df[num_cols].describe().round(2))

print("Lifetime < 0:", (df['Lifetime'] < 0).sum())
print("Qolgan oy > shartnoma muddati:", (df["Month_to_end_contract"] > df["Contract_period"]).sum())

Dublikatlar: 0
Bo'sh qiyamatlar: 0
gender [np.int64(0), np.int64(1)]
Near_Location [np.int64(0), np.int64(1)]
Partner [np.int64(0), np.int64(1)]
Promo_friends [np.int64(0), np.int64(1)]
Phone [np.int64(0), np.int64(1)]
Group_visits [np.int64(0), np.int64(1)]
Churn [np.int64(0), np.int64(1)]
           Age  Lifetime  Contract_period  Month_to_end_contract  \
count  4000.00   4000.00          4000.00                4000.00   
mean     29.18      3.72             4.68                   4.32   
std       3.26      3.75             4.55                   4.19   
min      18.00      0.00             1.00                   1.00   
25%      27.00      1.00             1.00                   1.00   
50%      29.00      3.00             1.00                   1.00   
75%      31.00      5.00             6.00                   6.00   
max      41.00     31.00            12.00                  12.00   

       Avg_additional_charges_total  Avg_class_frequency_total  \
count                       4

In [10]:
bin_fetats = [c for c in binary_cols if c != "Churn"]

print(df.groupby('Churn')[num_cols].mean().round(2).T)
print(df.groupby("Churn")[bin_fetats].mean().round(2).T)
print(df.groupby("Contract_period")["Churn"].agg(["count", 'mean']).round(2))
print(df.groupby(pd.cut(df['Lifetime'], [-1, 0, 1, 3, 6, 12, 40]))["Churn"].mean().round(2))

Churn                                   0       1
Age                                 29.98   26.99
Lifetime                             4.71    0.99
Contract_period                      5.75    1.73
Month_to_end_contract                5.28    1.66
Avg_additional_charges_total       158.45  115.08
Avg_class_frequency_total            2.02    1.47
Avg_class_frequency_current_month    2.03    1.04
Churn             0     1
gender         0.51  0.51
Near_Location  0.87  0.77
Partner        0.53  0.36
Promo_friends  0.35  0.18
Phone          0.90  0.90
Group_visits   0.46  0.27
                 count  mean
Contract_period             
1                 2207  0.42
6                  833  0.12
12                 960  0.02
Lifetime
(-1, 0]     0.83
(0, 1]      0.49
(1, 3]      0.19
(3, 6]      0.04
(6, 12]     0.00
(12, 40]    0.00
Name: Churn, dtype: float64


/tmp/ipykernel_1508/916211128.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(pd.cut(df['Lifetime'], [-1, 0, 1, 3, 6, 12, 40]))["Churn"].mean().round(2))


In [11]:
print(df.corr()['Churn'].drop('Churn').round(2).sort_values())

Lifetime                            -0.44
Avg_class_frequency_current_month   -0.41
Age                                 -0.40
Contract_period                     -0.39
Month_to_end_contract               -0.38
Avg_class_frequency_total           -0.25
Avg_additional_charges_total        -0.20
Group_visits                        -0.18
Promo_friends                       -0.16
Partner                             -0.16
Near_Location                       -0.13
gender                               0.00
Phone                               -0.00
Name: Churn, dtype: float64


In [12]:
print(df["Contract_period"].corr(df["Month_to_end_contract"]).round(2))

0.97


In [13]:
from sklearn.model_selection import train_test_split

Feats_all = [c for c in df.columns if c not in ['Churn', 'gender', 'Phone']]
Feats_safe = [c for c in Feats_all if c != "Avg_class_frequency_current_month"]

y = df['Churn']
X_tr, X_te, y_tr, y_te = train_test_split(df[Feats_all], y, test_size=0.2, stratify=y, random_state=42)

print(X_tr.shape, X_te.shape)
print(y_tr.mean().round(3), y_te.mean().round(3))

(3200, 11) (800, 11)
0.265 0.265


In [14]:
K = 160

def precision_recall_at_k(y_true, score, k):
  order = np.argsort(-np.asarray(score), kind='stable')[:k]
  hit = np.asarray(y_true)[order].sum()
  return hit / k, hit / np.sum(y_true)

rng = np.random.default_rng(42)
p, r = precision_recall_at_k(y_te, rng.random(len(y_te)), K)
print("Taasodifiy ro'yxat: precision", round(p, 3), "| recall", round(r, 3))

p, r = precision_recall_at_k(y_te, -X_te["Lifetime"], K)
print("Eng yangi a'zolar: precision", round(p, 3), "| recall", round(r, 3))

Taasodifiy ro'yxat: precision 0.269 | recall 0.203
Eng yangi a'zolar: precision 0.719 | recall 0.542


In [15]:
print((X_te["Lifetime"] <= 1).sum())

275


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score
from sklearn.ensemble import RandomForestClassifier

for name, feats in [("hammasi", Feats_all), ('current_month siz', Feats_safe)]:
  m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
  m.fit(X_tr[feats], y_tr)
  proba = m.predict_proba(X_te[feats])[:, 1]
  p, r = precision_recall_at_k(y_te, proba, K)
  print(name, "| precision", round(p, 3), "| recall", round(r, 3), "| PR-AUC", round(average_precision_score(y_te, proba), 3))

hammasi | precision 0.969 | recall 0.731 | PR-AUC 0.952
current_month siz | precision 0.9 | recall 0.679 | PR-AUC 0.899


In [17]:
models = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "rf": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42)
}
for name, mdl in models.items():
  scores = []
  for seed in [1, 2, 3, 4, 5]:
    a, b, ya, yb = train_test_split(X_tr[Feats_safe], y_tr, test_size=0.25, stratify=y_tr, random_state=seed)

    mdl.fit(a, ya)
    scores.append(average_precision_score(yb, mdl.predict_proba(b)[:, 1]))
    s = np.array(scores)
    print(name, s.round(3), "| o'rtacha", s.mean().round(3))

logreg [0.916] | o'rtacha 0.916
logreg [0.916 0.874] | o'rtacha 0.895
logreg [0.916 0.874 0.867] | o'rtacha 0.886
logreg [0.916 0.874 0.867 0.901] | o'rtacha 0.89
logreg [0.916 0.874 0.867 0.901 0.891] | o'rtacha 0.89
rf [0.9] | o'rtacha 0.9
rf [0.9   0.881] | o'rtacha 0.891
rf [0.9   0.881 0.866] | o'rtacha 0.882
rf [0.9   0.881 0.866 0.887] | o'rtacha 0.884
rf [0.9   0.881 0.866 0.887 0.894] | o'rtacha 0.886


In [18]:
for name, mdl in models.items():
  mdl.fit(X_tr[Feats_safe], y_tr)
  proba = mdl.predict_proba(X_te[Feats_safe])[:, 1]
  p, r = precision_recall_at_k(y_te, proba, K)
  print(name, "| precision", round(p, 3), "| recall", round(r, 3), "| PR-AUC", round(average_precision_score(y_te, proba), 3))

logreg | precision 0.9 | recall 0.679 | PR-AUC 0.899
rf | precision 0.912 | recall 0.689 | PR-AUC 0.882


In [19]:
lr = models['logreg']
coef = pd.Series(lr[-1].coef_[0], index=Feats_safe).sort_values()
print(coef.round(2))

Lifetime                       -3.75
Age                            -1.16
Contract_period                -0.73
Avg_class_frequency_total      -0.69
Month_to_end_contract          -0.65
Avg_additional_charges_total   -0.55
Group_visits                   -0.35
Promo_friends                  -0.19
Near_Location                  -0.08
Partner                        -0.02
dtype: float64


In [21]:
feats_nolife = [c for c in Feats_safe if c != "Lifetime"]
m2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
m2.fit(X_tr[feats_nolife], y_tr)
proba2 = m2.predict_proba(X_te[feats_nolife])[:, 1]
p, r = precision_recall_at_k(y_te, proba2, K)
print("Lifetimeasiz | precision", round(p, 3), "| recall", round(r, 3), "| PR-AUC", round(average_precision_score(y_te, proba2), 3))

Lifetimeasiz | precision 0.756 | recall 0.571 | PR-AUC 0.765


In [23]:
final = models['logreg']
final.fit(X_tr[Feats_safe], y_tr)
risk = pd.Series(final.predict_proba(X_te[Feats_safe])[:, 1], index=X_te.index)

tiers = pd.cut(risk.rank(ascending=False), [0, 60, 160, len(risk)], labels=['yuqori', "o'rta", "past"])
check = pd.DataFrame({"daraja": tiers, "kredit": y_te})
print(check.groupby("daraja", observed=True)['kredit'].agg(['count', 'mean', 'sum']).round(2))

        count  mean  sum
daraja                  
yuqori     60  0.97   58
o'rta     100  0.86   86
past      640  0.11   68


In [24]:
out = X_te[Feats_safe].copy()
out['risk'] = risk
out['daraja'] = tiers
out["kredit"] = y_te
out.to_csv("test_predictions.csv", index=False)

from google.colab import files
files.download("test_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>